# Statistical mechanics of the assembly substrate

Every figure here is generated from the tidy CSVs in `../experiments/`, through the
single shared analysis layer in `analysis.py`. Nothing is re-derived in this notebook,
on purpose: three copies of the same aggregation drifting apart is how two plots of
one dataset end up disagreeing with nobody able to say which is right.

## What is being measured

One coupling constant and four control parameters:

| symbol | meaning |
|---|---|
| $g=(1+\beta)^T$ | **gain** — $\beta$ and $T$ enter the dynamics only through this product |
| $\alpha = Mk/n$ | **load** — fraction of the substrate committed |
| $kp$ | **afferent count** — mean inputs from one assembly |
| $D$ | **depth** — levels of composition |

and two macrostates: retrieval accuracy $R$, and normalised overlap
$Q=(\bar q - q_0)/(1-q_0)$ with floor $q_0=k/n$.

## Reading these plots honestly

- Most of the campaign ran on **2 seeds**. Bootstrap intervals are drawn only where
  ≥3 seeds exist; elsewhere the marker is shown *without* a band rather than with a
  fake one. Check the seed census in the first cell before believing any curve.
- The working region is a **wedge** (starvation below, crowding above), so a single
  boundary is never the whole story. Widths are only reported where the grid
  actually brackets both walls.
- This is a **crossover, not a critical point**: the transition width does not shrink
  with $n$ over the measured range.

In [ ]:
import os, sys, math
sys.path.insert(0, os.path.abspath('../experiments'))

import numpy as np
import matplotlib.pyplot as plt
import analysis as A

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.22, 'grid.linewidth': 0.6,
    'legend.frameon': False, 'lines.linewidth': 1.8, 'lines.markersize': 4.5,
})
FIGDIR = os.path.abspath('../experiments/figures')
os.makedirs(FIGDIR, exist_ok=True)

rows = A.load()
print(f'{len(rows)} deepest-level rows from {sorted({r["source"] for r in rows})}')
print('\nSEED CENSUS  {n_seeds: n_configs} -- 2-seed cells cannot carry a CI')
print(A.seed_counts(rows))

## 1. The phase diagram

Retrieval over (load, gain) at $n=4000$, $D=5$, $kp=2.5$. The boundary is drawn
explicitly rather than left to be inferred from a colour legend.

This is the artefact the grammar work needs: it turns a ladder cell from an isolated
pass/fail into a **coordinate**.

In [ ]:
pa = [r for r in rows if r.get('cut') == 'gain_x_alpha']
alphas = sorted({r['alpha'] for r in pa})
gains  = sorted({r['gain'] for r in pa})
Z = np.full((len(gains), len(alphas)), np.nan)
grid = A.group(pa, ('alpha', 'gain'), 'acc')
for i, g in enumerate(gains):
    for j, a in enumerate(alphas):
        v = grid.get((a, g))
        if v: Z[i, j] = np.mean(v)

fig, ax = plt.subplots(figsize=(6.4, 4.6))
im = ax.pcolormesh(np.arange(len(alphas)+1), np.arange(len(gains)+1), Z,
                   cmap='RdYlBu', vmin=0, vmax=1, shading='flat')
fig.colorbar(im, ax=ax).set_label('retrieval accuracy $R$')

# Both walls, per load, from the shared wedge routine.
lo_x, lo_y, hi_x, hi_y = [], [], [], []
for j, a in enumerate(alphas):
    pts = [(g, np.mean(grid[(a, g)]), 0, 0, 0) for g in gains if (a, g) in grid]
    lo, hi, w, peak, gp, ok, note = A.wedge(pts)
    ypos = lambda v: np.interp(v, gains, np.arange(len(gains)) + 0.5)
    if lo == lo: lo_x.append(j + 0.5); lo_y.append(ypos(lo))
    if hi == hi: hi_x.append(j + 0.5); hi_y.append(ypos(hi))
    print(f'alpha={a:<5g} wedge=[{lo:.3f}, {hi:.3f}] width={w:.3f} peak={peak:.3f}'
          + ('' if ok else f'   <-- {note}'))
ax.plot(hi_x, hi_y, 'k-o', lw=2, ms=5, label='crowding wall')
ax.plot(lo_x, lo_y, 'k--s', lw=2, ms=5, label='starvation wall')
ax.legend(loc='center left')
ax.set_xticks(np.arange(len(alphas)) + 0.5); ax.set_xticklabels([f'{a:g}' for a in alphas])
ax.set_yticks(np.arange(len(gains)) + 0.5);  ax.set_yticklabels([f'{g:g}' for g in gains])
ax.set_xlabel(r'load  $\alpha = Mk/n$'); ax.set_ylabel(r'gain  $g=(1+\beta)^T$')
ax.set_title('Phase diagram: the working region is a wedge, and it closes')
ax.grid(False)
fig.savefig(f'{FIGDIR}/nb1_phase_diagram.png'); plt.show()

## 2. Wedge width against load — and where it closes

The lower wall rises and the upper wall falls, so the region closes **from both
sides**. This is the mechanism behind capacity: the substrate does not fail because
a boundary drifts past a fixed operating point, it fails because no gain works.

In [ ]:
ws = []
for a in alphas:
    pts = [(g, np.mean(grid[(a, g)]), 0, 0, 0) for g in gains if (a, g) in grid]
    lo, hi, w, peak, gp, ok, note = A.wedge(pts)
    if w == w and ok: ws.append((a, w))

fig, ax = plt.subplots(figsize=(5.0, 3.6))
if ws:
    ax.plot([a for a, _ in ws], [w for _, w in ws], 'o-', color='#B4436C')
    if len(ws) >= 2:
        a0, b0, *_ = A.fit_loglinear([a for a, _ in ws], [w for _, w in ws])
        xs = np.linspace(min(a for a, _ in ws) * 0.8, math.exp(-a0 / b0) * 1.05, 100)
        ax.plot(xs, a0 + b0 * np.log(xs), ':', color='0.5',
                label=f'$w={a0:.2f}{b0:+.2f}\\ln\\alpha$')
        star = math.exp(-a0 / b0)
        ax.axvline(star, color='k', ls='--', lw=1)
        ax.annotate(f'closes at $\\alpha^*\\approx{star:.2f}$', (star, 0.05),
                    xytext=(-8, 0), textcoords='offset points', ha='right', fontsize=8)
        print(f'extrapolated closure: alpha* = {star:.3f}')
ax.axhline(0, color='0.7', lw=0.8)
ax.set_xlabel(r'load  $\alpha$'); ax.set_ylabel('wedge width in gain')
ax.set_title('The usable band narrows to nothing')
ax.legend()
fig.savefig(f'{FIGDIR}/nb2_wedge_width.png'); plt.show()

## 3. Where the boundary sits: four axes

$g_c$ depends on **all four** of $n$, $\alpha$, $D$ and $k$. That is the source of the
single most costly methodological error in this campaign, which recurred three times:

> Whenever $g_c$ depends on an axis, comparing along that axis at **fixed absolute
> gain** is confounded.

It invalidated a superlinear capacity exponent, an apparent invariance in $k$, and
produced one outlier that looked like a refutation.

In [ ]:
def gc_of(subset, field):
    g = A.group(subset, (field, 'gain'), 'acc')
    xs = sorted({k[0] for k in g})
    gains_l = sorted({k[1] for k in g})
    out = []
    for x in xs:
        pts = [(gv, np.mean(g[(x, gv)]), 0, 0, 0) for gv in gains_l if (x, gv) in g]
        lo, hi, w, peak, gp, ok, note = A.wedge(pts)
        if hi == hi: out.append((x, hi))
    return out

panels = [
    ('system size $n$', gc_of([r for r in rows if r['source']=='critical_point_scan.csv'], 'n'), True),
    (r'load $\alpha$',  gc_of(pa, 'alpha'), True),
    ('depth $D$',       gc_of([r for r in rows if r.get('cut')=='gain_x_depth'], 'depth'), False),
    ('assembly size $k$', gc_of([r for r in rows if r.get('cut')=='gain_x_k'], 'k'), True),
]
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.2))
for ax, (label, pts, logx) in zip(axes, panels):
    if not pts:
        ax.text(.5, .5, 'no data', ha='center', transform=ax.transAxes); ax.set_title(label); continue
    ax.plot([p[0] for p in pts], [p[1] for p in pts], 'o-', color='#2E6E8E')
    for x, y in pts:
        ax.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(5, -3), fontsize=7)
    if logx:
        ax.set_xscale('log'); ax.set_xticks([p[0] for p in pts])
        ax.set_xticklabels([f'{p[0]:g}' for p in pts])
    ax.set_xlabel(label); ax.set_ylabel('crowding wall $g_c$')
fig.suptitle('$g_c$ moves along every axis — so fixed-absolute-gain comparisons are confounded', y=1.04)
fig.savefig(f'{FIGDIR}/nb3_gc_four_axes.png'); plt.show()

## 4. Capacity is extensive

Measured at fixed **relative** gain $g=0.88\,g_c(n)$ — the only comparison that means
anything once $g_c(n)$ is known to move. An earlier $n^{1.49}$ result taken at fixed
*absolute* gain is **withdrawn**: it inherited the boundary's movement.

$$M_{\max}\;\approx\;1.15\,\frac{n}{k}$$

In [ ]:
cr = [r for r in rows if r.get('cut') == 'capacity_rel']
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
cmap = plt.get_cmap('viridis'); sizes = sorted({r['n'] for r in cr}); mm = []
for i, n in enumerate(sizes):
    cur = A.curve([r for r in cr if r['n'] == n], 'M', 'acc')
    pts = cur.get((), [])
    if not pts: continue
    c = cmap(i / max(len(sizes)-1, 1) * 0.85)
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    axes[0].plot(xs, ys, 'o-', color=c, label=f'n={n:,}')
    if all(p[2] == p[2] for p in pts):
        axes[0].fill_between(xs, [p[2] for p in pts], [p[3] for p in pts], color=c, alpha=.15, lw=0)
    x = A.cross_down(pts)
    if x == x: mm.append((n, x)); axes[0].axvline(x, color=c, ls=':', lw=1)
axes[0].axhline(.5, color='0.6', ls=':', lw=.8)
axes[0].set_xscale('log', base=2); axes[0].set_xlabel('items stored $M$')
axes[0].set_ylabel('retrieval $R$'); axes[0].legend()
axes[0].set_title(r'Capacity at fixed relative gain $0.88\,g_c(n)$')

if len(mm) >= 2:
    ns = [m[0] for m in mm]; ms = [m[1] for m in mm]
    axes[1].plot(ns, ms, 'o-', color='#B4436C', label='measured')
    ref = [ms[0] * (n / ns[0]) for n in ns]
    axes[1].plot(ns, ref, '--', color='0.5', label='extensive ($\\propto n$)')
    axes[1].set_xscale('log'); axes[1].set_yscale('log')
    axes[1].set_xticks(ns); axes[1].set_xticklabels([f'{n:,}' for n in ns])
    axes[1].set_xlabel('$n$'); axes[1].set_ylabel('$M_{\\max}$'); axes[1].legend()
    expo = math.log(ms[-1]/ms[0]) / math.log(ns[-1]/ns[0])
    axes[1].set_title(f'exponent {expo:.2f}  (1.0 = extensive)')
    for n, m in mm: print(f'n={n:<6} M_max={m:6.1f}   alpha*={m*50/n:.3f}')
fig.savefig(f'{FIGDIR}/nb4_capacity.png'); plt.show()

## 5. The information budget

Fano's inequality turns measured retrieval into a floor under the bits surviving at
each level; the data-processing inequality guarantees the true quantity is
non-increasing along the chain. Bits are comparable across $M$ and $n$ in a way
accuracy is not.

The shape is a **cliff, not a slope** — so an averaged "bits lost per level" would
describe a decay that does not happen.

In [ ]:
def h2(x): return 0.0 if x <= 0 or x >= 1 else -(x*math.log2(x) + (1-x)*math.log2(1-x))
def fano(acc, M):
    pe = max(0.0, min(1.0, 1-acc))
    return max(0.0, math.log2(M) - h2(pe) - pe*math.log2(max(M-1, 1)))

deep = A.load(deepest_only=False)
sel = [r for r in deep if r.get('cut') == 'capacity_rel' and r['n'] == 2000]
fig, ax = plt.subplots(figsize=(5.4, 3.8))
cmap = plt.get_cmap('magma'); Ms = sorted({r['M'] for r in sel}); usable = []
for i, M in enumerate(Ms):
    lv = sorted({r['level'] for r in sel if r['M'] == M})
    bits = [np.mean([fano(r['acc'], M) for r in sel if r['M']==M and r['level']==L]) for L in lv]
    ax.plot(lv, bits, 'o-', color=cmap(0.15 + 0.7*i/max(len(Ms)-1,1)), label=f'M={M}')
    u = 0
    for L, b in zip(lv, bits):
        if b >= 0.5*math.log2(M): u = L
        else: break
    usable.append((M, u))
ax.set_xlabel('composition level'); ax.set_ylabel('bits retained (Fano floor)')
ax.set_title('Information holds, then falls off a cliff  ($n=2000$)')
ax.legend(fontsize=7, ncol=2)
fig.savefig(f'{FIGDIR}/nb5_information.png'); plt.show()

print('usable depth by vocabulary:', usable)
pts = [(M, u) for M, u in usable if u > 0]
if len(pts) >= 2 and pts[0][1] > pts[-1][1]:
    cost = (pts[0][1]-pts[-1][1]) / math.log2(pts[-1][0]/pts[0][0])
    print(f'==> {cost:.2f} levels of depth lost per DOUBLING of vocabulary')

## 6. Zipfian vocabulary: one area, two incompatible demands

Plasticity is cumulative, so under Zipf the *effective* gain of an item is $g^{c_i}$
where $c_i$ is its frequency. The gain stops being a control parameter and becomes a
**distribution over items** — frequent items pushed toward the crowding wall, rare
items left against the starvation wall, in the same area at the same time.

Note the control: at skew 0 each item is merged exactly once, which *is* the standard
protocol used everywhere else, so the uniform arm reproduces the phase diagram.

In [ ]:
z = [r for r in rows if r.get('cut') == 'zipf']
if not z:
    print('no zipf data yet -- run experiments/zipf_grammar.py')
else:
    skews = sorted({r['zipf_s'] for r in z})
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.8))
    cmap = plt.get_cmap('coolwarm')
    for i, s in enumerate(skews):
        sub = [r for r in z if r['zipf_s'] == s]
        c = cmap(i / max(len(skews)-1, 1))
        for ax, field, style in ((axes[0], 'acc_head', '-o'), (axes[0], 'acc_tail', '--s')):
            pts = A.curve(sub, 'gain', field).get((), [])
            if pts:
                ax.plot([p[0] for p in pts], [p[1] for p in pts], style, color=c,
                        label=f's={s:g} {field.split("_")[1]}')
        pts = A.curve(sub, 'gain', 'acc').get((), [])
        if pts:
            xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
            axes[1].plot(xs, ys, 'o-', color=c, label=f's={s:g}')
            if all(p[2]==p[2] for p in pts):
                axes[1].fill_between(xs, [p[2] for p in pts], [p[3] for p in pts],
                                     color=c, alpha=.15, lw=0)
    axes[0].set_xlabel('gain $g$ (per merge)'); axes[0].set_ylabel('retrieval $R$')
    axes[0].set_title('Head (frequent) vs tail (rare)'); axes[0].legend(fontsize=6, ncol=2)
    axes[1].axhline(.5, color='0.6', ls=':', lw=.8)
    axes[1].set_xlabel('gain $g$ (per merge)'); axes[1].set_ylabel('retrieval $R$')
    axes[1].set_title('Overall, by skew (bands = bootstrap CI)'); axes[1].legend(fontsize=7)
    fig.savefig(f'{FIGDIR}/nb6_zipf.png'); plt.show()

    print(f"{'skew':>6} {'wedge':>18} {'width':>7}   head-vs-tail gap at peak")
    for s in skews:
        sub = [r for r in z if r['zipf_s'] == s]
        pts = A.curve(sub, 'gain', 'acc').get((), [])
        lo, hi, w, peak, gp, ok, note = A.wedge(pts)
        hd = np.mean([r['acc_head'] for r in sub if r['gain'] == gp])
        tl = np.mean([r['acc_tail'] for r in sub if r['gain'] == gp])
        print(f'{s:>6.2f} [{lo:6.3f},{hi:6.3f}] {w:>7.3f}   '
              f'head {hd:.3f} vs tail {tl:.3f}  ({hd-tl:+.3f})'
              + ('' if ok else f'  <-- {note}'))

## 7. Bridge: does the map predict the real parser?

The map was built on a synthetic task, so it may simply describe a different system.
This is the test against `NemoParser`, which uses `rounds=10` — meaning ONE training
step applies $(1.1)^{10}=2.594$ at $beta=0.1$, already past a wall near 1.9, and
cumulatively $1.1^{10c}=17.4$ at $c_{\max}=3$.

**Two factors are required.** Varying $beta$ alone cannot separate *the map is right*
from *a structural bug pins these areas* — both produce collapse. `train_roles` calls
`reset_area_connections` after every word, which zeroes the connectome so $k$-WTA falls
through to its index tie-break and returns identical winners regardless of $beta$.

That is why the first pass looked like a confirmation and was worthless: distinctness
0.333 and spread 1.0000 at *every* $beta$ across a 25× sweep. **A gain effect that is
invariant to gain is not a gain effect.**

In [ ]:
br = [r for r in A.load(deepest_only=False) if r.get('cut') == 'bridge']
if not br:
    print('no bridge data -- run experiments/bridge_parser.py')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), sharey=True)
    floor = br[0]['floor']
    for ax, reset in zip(axes, (0, 1)):
        sub = [r for r in br if r['reset'] == reset]
        for kind, colour, mark in (('role', '#B4436C', 'o'), ('lexicon', '#2E6E8E', 's')):
            pts = A.curve([r for r in sub if r['kind'] == kind], 'beta', 'spread')
            p = pts.get((), [])
            if not p: continue
            xs = [q[0] for q in p]; ys = [q[1] for q in p]
            ax.plot(xs, ys, mark + '-', color=colour, label=f'{kind} areas')
            if all(q[2] == q[2] for q in p):
                ax.fill_between(xs, [q[2] for q in p], [q[3] for q in p],
                                color=colour, alpha=.15, lw=0)
        ax.axhline(floor, color='0.4', ls='--', lw=1)
        ax.annotate('chance floor $k/n$', (min(r["beta"] for r in sub), floor),
                    xytext=(2, 4), textcoords='offset points', fontsize=7, color='0.35')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('plasticity  beta')
        ax.set_title('reset AS SHIPPED' if reset == 0 else 'reset NEUTRALISED')
        ax.legend(fontsize=7)
    axes[0].set_ylabel('mean pairwise overlap')
    fig.suptitle('Bridge: overlap vs plasticity in the real parser', y=1.03)
    fig.savefig(f'{FIGDIR}/nb7_bridge.png'); plt.show()

    print(f"{'reset':>6} {'beta':>8} {'g/step':>8} {'kind':>8} {'distinct':>9} {'spread':>8}")
    seen = set()
    for r in sorted(br, key=lambda r: (r['reset'], r['beta'], r['kind'])):
        key = (r['reset'], r['beta'], r['kind'])
        if key in seen: continue
        seen.add(key)
        same = [x for x in br if (x['reset'], x['beta'], x['kind']) == key]
        d = np.mean([x['distinct_frac'] for x in same])
        s = np.mean([x['spread'] for x in same])
        print(f"{r['reset']:>6} {r['beta']:>8.4f} {r['g_step']:>8.3f} "
              f"{r['kind']:>8} {d:>9.3f} {s:>8.4f}")

## Established laws, with status

| claim | status |
|---|---|
| $\beta,T$ act only through $g=(1+\beta)^T$ | **measured**, $\beta\in[0.18,0.45]$ at fixed boundary |
| Working region is a wedge; closes under load and depth | **measured** |
| $M_{\max}\approx1.15\,n/k$ (capacity extensive) | **measured**, exponent 1.01 over 4× |
| $g_c$ rises with $n$, falls with $\alpha$, $D$, $k$ | **measured** |
| $kp\lesssim1$ closes the wedge outright | **measured** |
| ~1.66 depth levels lost per vocabulary doubling | **measured** at $n=2000$ |
| Crossover, not a critical point | **measured**, limited by grid resolution |
| Derivation of $g_c$ from order statistics | **open** |
| Optimal gain falls with depth | **open** — weakest claim on the board |

### Limits worth restating

Two seeds over most of the grid; one protocol ($T=2$, gated merge, chain of shared
areas) — so this may be *this protocol's* phase diagram rather than the substrate's;
and until the Zipf arm, a synthetic uniform task where real grammar is skewed,
reuses constituents, and is hierarchical rather than chained.

The unfinished milestone is the **bridge experiment**: measure the parser's real
$(\alpha, g, kp, D)$ per area and test whether this map predicts its behaviour. If it
does, grammars can be scaled by construction; if not, the map is about the wrong
system, and one experiment is a cheap way to find that out.